# 20. Within-HASOC replication

Dr Hunter has asked for this twice, in different words. The question is whether the collapse
on the hate class is a property of the Bohra corpus or a property of the task.

Every result so far trains on Bohra. If a model trained and tested **inside HASOC 2022**
shows the same pattern, the finding generalises across corpora and the dissertation's claim
is about Hinglish hate-speech detection. If it does not, the finding is about Bohra
specifically, which is still publishable but is a different and narrower claim.

It also separates two explanations that are currently tangled together. The cross-dataset
collapse reported in Chapter 4 could be caused either by the shift between corpora or by
HASOC 2022 simply being harder. Training inside HASOC answers that.

**Runtime: under a minute, CPU only**, for the statistical baseline. A transformer section is
included and needs a GPU.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'notebooks'))
DATA, W = ROOT / 'data', ROOT / 'writing'
SEED = 42

from hinglish_hate import load_hasoc2022_threads, filter_romanised, load_lexicon
from hinglish_hate.baseline import build_full_pipeline

h22 = filter_romanised(load_hasoc2022_threads(DATA), include_mixed=True).reset_index(drop=True)
print(f'HASOC 2022 romanised: {len(h22)} posts, {100*h22.label.mean():.1f}% hate')

tr22, te22 = train_test_split(h22, test_size=0.2, stratify=h22['label'], random_state=SEED)
print(f'split: {len(tr22)} train / {len(te22)} test')
print(f'(Bohra for comparison: 3659 / 915, 36.3% hate)')

In [ ]:
terms = load_lexicon(DATA / 'hinglish_slur_lexicon.csv')

def evaluate(name, tr, te):
    pipe = build_full_pipeline(terms)
    pipe.fit(tr['text'], tr['label'])
    pte = pipe.predict(te['text'])
    ptr = pipe.predict(tr['text'])
    tn, fp, fn, tp = confusion_matrix(te['label'], pte, labels=[0, 1]).ravel()
    return {'setting': name,
            'n_train': len(tr), 'n_test': len(te),
            'hate_rate': round(float(tr['label'].mean()), 3),
            'train_acc': round(accuracy_score(tr['label'], ptr), 3),
            'test_acc': round(accuracy_score(te['label'], pte), 3),
            'macro_f1': round(f1_score(te['label'], pte, average='macro'), 3),
            'hate_f1': round(f1_score(te['label'], pte, pos_label=1, zero_division=0), 3),
            'hate_recall': round(tp / max(tp + fn, 1), 3),
            'not_recall': round(tn / max(tn + fp, 1), 3)}

rows = []
t0 = time.time()

# within-HASOC
rows.append(evaluate('HASOC 2022 -> HASOC 2022 (within)', tr22, te22))

# within-Bohra, the existing reference
trB = pd.read_parquet(DATA / 'splits' / 'bohra_train.parquet')
teB = pd.read_parquet(DATA / 'splits' / 'bohra_test.parquet')
rows.append(evaluate('Bohra -> Bohra (within)', trB, teB))

# the two cross directions
rows.append(evaluate('Bohra -> HASOC 2022 (cross)', trB, te22))
rows.append(evaluate('HASOC 2022 -> Bohra (cross)', tr22, teB))

res = pd.DataFrame(rows)
res.to_csv(W / 'within_hasoc_replication.csv', index=False)
print(f'{time.time()-t0:.0f}s\n')
display(res)

In [ ]:
# five-fold cross-validation inside HASOC, so the within number is not one lucky split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores, hate_f1s, recalls = [], [], []
for k, (i_tr, i_te) in enumerate(skf.split(h22['text'], h22['label']), 1):
    a, b = h22.iloc[i_tr], h22.iloc[i_te]
    pipe = build_full_pipeline(terms)
    pipe.fit(a['text'], a['label'])
    p = pipe.predict(b['text'])
    tn, fp, fn, tp = confusion_matrix(b['label'], p, labels=[0, 1]).ravel()
    scores.append(f1_score(b['label'], p, average='macro'))
    hate_f1s.append(f1_score(b['label'], p, pos_label=1, zero_division=0))
    recalls.append(tp / max(tp + fn, 1))
    print(f'  fold {k}: macro-F1 {scores[-1]:.3f}  hate-F1 {hate_f1s[-1]:.3f}  '
          f'hate recall {recalls[-1]:.3f}')

print(f'\nHASOC 2022, 5-fold: macro-F1 {np.mean(scores):.3f} +/- {np.std(scores):.3f}'
      f' | hate recall {np.mean(recalls):.3f}')
print(f'Bohra,      5-fold: macro-F1 0.657 +/- 0.005 (notebook 07)')

pd.DataFrame({'fold': range(1, 6), 'macro_f1': np.round(scores, 4),
              'hate_f1': np.round(hate_f1s, 4), 'hate_recall': np.round(recalls, 4)}
            ).to_csv(W / 'hasoc_cv_folds.csv', index=False)

## Reading it

The comparison that matters is **hate recall within each corpus**. On Bohra it is about 0.58.

- **If HASOC within-corpus hate recall is similarly low**, the difficulty is a property of
  Hinglish hate-speech detection, not of one dataset, and the dissertation's claim is the
  broader one.
- **If it is markedly higher**, Bohra is the harder or noisier corpus, the cross-dataset
  collapse reported earlier is partly a Bohra artefact, and that is worth saying plainly
  because it is a critique of a corpus the field still uses.

The two cross rows also give something the dissertation does not yet have: transfer measured
in **both** directions. If Bohra to HASOC collapses but HASOC to Bohra does not, the asymmetry
is itself a result, and it points at the label definitions rather than at domain shift.

In [ ]:
RUN_TRANSFORMER = False      # set True on the A5000; about 4 minutes

if RUN_TRANSFORMER:
    import torch, random, gc
    from torch.utils.data import Dataset, DataLoader
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              get_linear_schedule_with_warmup)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    MODEL, EPOCHS, LR, BS = 'xlm-roberta-base', 3, 2e-5, 16

    class PostDataset(Dataset):
        def __init__(self, texts, labels, tok, max_len=128):
            self.t = list(texts); self.y = list(labels); self.tok = tok; self.m = max_len
        def __len__(self): return len(self.t)
        def __getitem__(self, i):
            e = self.tok(str(self.t[i]), truncation=True, padding='max_length',
                         max_length=self.m, return_tensors='pt')
            return {'input_ids': e['input_ids'].squeeze(0),
                    'attention_mask': e['attention_mask'].squeeze(0),
                    'labels': torch.tensor(self.y[i], dtype=torch.long)}

    def set_all_seeds(s):
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

    set_all_seeds(SEED)
    tok = AutoTokenizer.from_pretrained(MODEL)
    dl = DataLoader(PostDataset(tr22['text'], tr22['label'], tok), batch_size=BS, shuffle=True)
    counts = np.bincount(tr22['label'].values, minlength=2)
    wts = torch.tensor(len(tr22) / (2.0 * np.maximum(counts, 1)),
                       dtype=torch.float).to(DEVICE)
    set_all_seeds(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    lossf = torch.nn.CrossEntropyLoss(weight=wts)
    for _ in range(EPOCHS):
        model.train()
        for b in dl:
            lg = model(input_ids=b['input_ids'].to(DEVICE),
                       attention_mask=b['attention_mask'].to(DEVICE)).logits
            loss = lossf(lg, b['labels'].to(DEVICE)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()

    def pred(frame):
        d = DataLoader(PostDataset(frame['text'], frame['label'], tok), batch_size=BS*2)
        model.eval(); P, T = [], []
        with torch.no_grad():
            for b in d:
                lg = model(input_ids=b['input_ids'].to(DEVICE),
                           attention_mask=b['attention_mask'].to(DEVICE)).logits
                P.extend(lg.argmax(1).cpu().numpy()); T.extend(b['labels'].numpy())
        return np.array(T), np.array(P)

    trows = []
    for nm, fr in [('HASOC -> HASOC (within), XLM-R', te22),
                   ('HASOC -> Bohra (cross), XLM-R', teB)]:
        y, p = pred(fr)
        tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
        trows.append({'setting': nm, 'macro_f1': round(f1_score(y, p, average='macro'), 3),
                      'hate_f1': round(f1_score(y, p, pos_label=1, zero_division=0), 3),
                      'hate_recall': round(tp / max(tp + fn, 1), 3),
                      'not_recall': round(tn / max(tn + fp, 1), 3)})
        print(trows[-1])
    pd.DataFrame(trows).to_csv(W / 'within_hasoc_xlmr.csv', index=False)
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print('transformer section skipped (RUN_TRANSFORMER = False)')